In [1]:
!pip install -q transformers datasets jiwer librosa soundfile evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.3 MB/s eta 0:00:0000:0100:01


In [2]:
import os, re, json, csv, unicodedata
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import urllib.request
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from datasets import Dataset, DatasetDict, Audio, load_dataset
import evaluate

In [7]:
SAMPLE_RATE   = 16000
MAX_DURATION  = 30.0        # Whisper max input = 30s
MIN_DURATION  = 1.0         # Skip very short segments
MODEL_NAME    = "openai/whisper-small"
LANGUAGE      = "Hindi"
TASK          = "transcribe"
CSV_PATH = "/kaggle/input/datasets/sivatejareddya/josh-talks-hindi-asr/FT Data - data.csv"

DATA_DIR  = Path("/kaggle/working/data")
AUDIO_DIR = DATA_DIR / "audio"
SEG_DIR   = DATA_DIR / "segments"
OUT_DIR   = Path("/kaggle/working/output")
for d in [AUDIO_DIR, SEG_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [8]:
# ── CELL 4: URL helper ───────────────────────────────────────────────────────
def fix_url(url: str) -> str:
    """Rewrite broken GCP URL to accessible bucket URL."""
    m = re.search(r'/hq_data/hi/(\d+)/(.+)$', url)
    return f'https://storage.googleapis.com/upload_goai/{m.group(1)}/{m.group(2)}' if m else url

In [9]:
# ── CELL 5: Download + Slice audio into segments ─────────────────────────────
def load_transcription(url: str) -> List[dict]:
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.loads(r.read())

def process_recording(row: dict) -> List[dict]:
    rid  = str(row['recording_id'])
    wav  = AUDIO_DIR / f"{rid}.wav"

    # Download full recording (skip if cached)
    if not wav.exists():
        try:
            urllib.request.urlretrieve(fix_url(row['rec_url_gcp']), wav)
        except Exception as e:
            print(f"  [SKIP] audio download failed: {e}")
            return []

    # Fetch transcription segments
    try:
        segs = load_transcription(fix_url(row['transcription_url_gcp']))
    except Exception as e:
        print(f"  [SKIP] transcription failed: {e}")
        return []

    # Load audio once, slice per segment
    try:
        audio, _ = librosa.load(wav, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f"  [SKIP] audio load failed: {e}")
        return []

    results = []
    for i, seg in enumerate(segs):
        start, end = float(seg.get('start', 0)), float(seg.get('end', 0))
        dur  = end - start
        text = seg.get('text', '').strip()

        if dur < MIN_DURATION or dur > MAX_DURATION or not text:
            continue

        seg_path = SEG_DIR / f"{rid}_{i:03d}.wav"
        if not seg_path.exists():
            sf.write(seg_path, audio[int(start*SAMPLE_RATE):int(end*SAMPLE_RATE)], SAMPLE_RATE)

        results.append({
            'audio_path'   : str(seg_path),
            'text'         : text,
            'recording_id' : rid,
            'segment_id'   : i,
            'duration'     : round(dur, 2),
        })
    return results

df = pd.read_csv(CSV_PATH)
print(f"Recordings: {len(df)}")

all_segs = []
for i, row in df.iterrows():
    print(f"[{i+1:3d}/{len(df)}] {row['recording_id']}", end=" → ")
    s = process_recording(row.to_dict())
    print(f"{len(s)} segs")
    all_segs.extend(s)

seg_df = pd.DataFrame(all_segs)
print(f"\nTotal segments : {len(seg_df)}")
print(f"Duration stats :\n{seg_df['duration'].describe()}")
seg_df.to_csv(OUT_DIR / "segments_manifest.csv", index=False)

Recordings: 104
[  1/104] 825780 → 20 segs
[  2/104] 825727 → 31 segs
[  3/104] 988596 → 25 segs
[  4/104] 990175 → 22 segs
[  5/104] 526266 → 32 segs
[  6/104] 520199 → 25 segs
[  7/104] 542785 → 41 segs
[  8/104] 494019 → 38 segs
[  9/104] 523045 → 43 segs
[ 10/104] 522951 → 32 segs
[ 11/104] 254219 → 51 segs
[ 12/104] 253253 → 43 segs
[ 13/104] 351501 → 36 segs
[ 14/104] 350606 → 43 segs
[ 15/104] 629904 → 42 segs
[ 16/104] 635909 → 48 segs
[ 17/104] 989901 → 85 segs
[ 18/104] 990783 → 95 segs
[ 19/104] 240907 → 36 segs
[ 20/104] 240909 → 23 segs
[ 21/104] 270153 → 40 segs
[ 22/104] 270150 → 46 segs
[ 23/104] 475392 → 35 segs
[ 24/104] 475356 → 35 segs
[ 25/104] 255349 → 53 segs
[ 26/104] 255381 → 51 segs
[ 27/104] 767685 → 42 segs
[ 28/104] 767869 → 59 segs
[ 29/104] 886193 → 69 segs
[ 30/104] 888331 → 77 segs
[ 31/104] 615351 → 28 segs
[ 32/104] 615319 → 22 segs
[ 33/104] 738845 → 80 segs
[ 34/104] 739630 → 88 segs
[ 35/104] 272241 → 72 segs
[ 36/104] 282447 → 87 segs
[ 37/104] 27

In [ ]:
# ── CELL 6: Train/Val Split ──────────────────────────────────────────────────
# Split by recording_id so no speaker leaks across splits
rec_ids = seg_df['recording_id'].unique()
np.random.seed(42)
np.random.shuffle(rec_ids)
split = int(len(rec_ids) * 0.9)
train_ids, val_ids = set(rec_ids[:split]), set(rec_ids[split:])

train_df = seg_df[seg_df['recording_id'].isin(train_ids)].reset_index(drop=True)
val_df   = seg_df[seg_df['recording_id'].isin(val_ids)].reset_index(drop=True)
print(f"Train: {len(train_df)} segs ({len(train_ids)} recordings)")
print(f"Val  : {len(val_df)} segs ({len(val_ids)} recordings)")

def make_ds(df: pd.DataFrame) -> Dataset:
    return (Dataset.from_dict({'audio': df['audio_path'].tolist(), 'text': df['text'].tolist()})
            .cast_column('audio', Audio(sampling_rate=SAMPLE_RATE)))

raw_ds = DatasetDict({'train': make_ds(train_df), 'validation': make_ds(val_df)})



In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer  = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor  = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

def prepare_sample(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = tokenizer(batch['text']).input_ids
    return batch

prepared_train = raw_ds["train"].map(
    prepare_sample,
    remove_columns=raw_ds["train"].column_names,
    desc="Extracting train features",
)
prepared_val = raw_ds["validation"].map(
    prepare_sample,
    remove_columns=raw_ds["validation"].column_names,
    desc="Extracting val features",
)
prepared_ds = DatasetDict({"train": prepared_train, "validation": prepared_val})

In [ ]:
# ── CELL 8: Data Collator ────────────────────────────────────────────────────
@dataclass
class SpeechSeq2SeqCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        inputs = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt"
        )
        labels_batch = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        inputs["labels"] = labels
        return inputs


In [ ]:

# ── CELL 9: Model + Training ─────────────────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task     = TASK
model.generation_config.forced_decoder_ids = None

collator   = SpeechSeq2SeqCollator(processor, model.config.decoder_start_token_id)
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred.predictions,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids,         skip_special_tokens=True)
    return {"wer": round(100 * wer_metric.compute(predictions=pred_str, references=label_str), 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "/kaggle/working/whisper-hindi",
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 1,
    learning_rate               = 1e-5,
    warmup_steps                = 200,
    max_steps                   = 1000,
    gradient_checkpointing      = True,
    fp16                        = True,
    eval_strategy               = "steps",
    per_device_eval_batch_size  = 8,
    predict_with_generate       = True,
    generation_max_length       = 225,
    save_steps                  = 250,
    eval_steps                  = 250,
    logging_steps               = 50,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
)

trainer = Seq2SeqTrainer(
    args               = training_args,
    model              = model,
    train_dataset      = prepared_ds["train"],
    eval_dataset       = prepared_ds["validation"],
    data_collator      = collator,
    compute_metrics    = compute_metrics,
    processing_class   = processor.feature_extractor,
)

print("Starting fine-tuning...")
trainer.train()
trainer.save_model("/kaggle/working/whisper-hindi-best")
processor.save_pretrained("/kaggle/working/whisper-hindi-best")
print("Training done.")



In [10]:
!pip install transformers torch librosa soundfile evaluate jiwer unicodedata2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 7.1 MB/s eta 0:00:00:00:0100:01


In [11]:
# =============================================================================
# Q1: Evaluation + Error Analysis (run locally after training)
# Uses the saved whisper-hindi-best model + segments_manifest.csv
# =============================================================================

# pip install transformers torch librosa soundfile evaluate jiwer unicodedata2

import re, json, unicodedata, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import evaluate

# ── Config ───────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 16000
MODEL_NAME    = "openai/whisper-small"
FINETUNED_DIR = "/kaggle/input/datasets/sivatejareddya/whisper-hindi-best/whisper-hindi-best"
MANIFEST_PATH = "/kaggle/input/datasets/sivatejareddya/josh-talks-segments/segments_manifest.csv"
CSV_PATH      = "/kaggle/input/datasets/sivatejareddya/josh-talks-hindi-asr/FT Data - data.csv"
OUT_DIR       = Path("/kaggle/working/output")
SEG_CACHE     = Path("/kaggle/working/data/segments")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEG_CACHE.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Load processor + metrics ─────────────────────────────────────────────────
processor  = WhisperProcessor.from_pretrained(FINETUNED_DIR, language="Hindi", task="transcribe")
wer_metric = evaluate.load("wer")

# ── URL helper ────────────────────────────────────────────────────────────────
def fix_url(url):
    m = re.search(r'/hq_data/hi/(\d+)/(.+)$', url)
    return f'https://storage.googleapis.com/upload_goai/{m.group(1)}/{m.group(2)}' if m else url

# ── Download + cache segment audio locally ───────────────────────────────────
# ── Load manifest + build val split (same seed as training) ──────────────────
seg_df  = pd.read_csv(MANIFEST_PATH)
rec_ids = seg_df['recording_id'].unique()
np.random.seed(42)
np.random.shuffle(rec_ids)
val_ids = set(rec_ids[int(len(rec_ids) * 0.9):])
val_df  = seg_df[seg_df['recording_id'].isin(val_ids)].reset_index(drop=True)
print(f"Val split: {len(val_df)} segments from {len(val_ids)} recordings")

# Verify audio files exist
missing = val_df[~val_df['audio_path'].apply(lambda p: Path(p).exists())]
print(f"Found: {len(val_df) - len(missing)} | Missing: {len(missing)}")
val_df = val_df[val_df['audio_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
print(f"Ready: {len(val_df)} segments")

# ── Inference helper ─────────────────────────────────────────────────────────
def infer(model, audio_path):
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    feats = processor.feature_extractor(
        audio, sampling_rate=SAMPLE_RATE, return_tensors="pt"
    ).input_features.to(DEVICE)
    with torch.no_grad():
        ids = model.generate(feats, language="hi", task="transcribe")
    return processor.tokenizer.decode(ids[0], skip_special_tokens=True)

def eval_model(model_path, label):
    print(f"\nEvaluating: {label}")
    m = WhisperForConditionalGeneration.from_pretrained(model_path).to(DEVICE).eval()
    preds, refs = [], []
    for i, row in val_df.iterrows():
        pred = infer(m, row['audio_path'])
        preds.append(pred)
        refs.append(row['text'])
        if (i+1) % 50 == 0:
            print(f"  {i+1}/{len(val_df)} done")
    wer = round(100 * wer_metric.compute(predictions=preds, references=refs), 2)
    print(f"  WER = {wer:.2f}%")
    return wer, preds, refs

# ── CELL 10: Evaluate ─────────────────────────────────────────────────────────
print("\n" + "="*55)
print("EVALUATION RESULTS")
print("="*55)
baseline_wer, baseline_preds, val_refs = eval_model(MODEL_NAME,    "Whisper-small (pretrained)")
ftune_wer,    ftune_preds,    _        = eval_model(FINETUNED_DIR, "Whisper-small (fine-tuned)")

print(f"\n{'Model':<40} {'WER (%)':>8}")
print("-"*55)
print(f"{'Whisper-small (pretrained)':<40} {baseline_wer:>8.2f}")
print(f"{'Whisper-small (fine-tuned on Hindi data)':<40} {ftune_wer:>8.2f}")
print(f"{'Absolute Improvement':<40} {baseline_wer - ftune_wer:>+8.2f}")
print("="*55)




Device: cuda


Val split: 565 segments from 11 recordings
Found: 565 | Missing: 0
Ready: 565 segments

EVALUATION RESULTS

Evaluating: Whisper-small (pretrained)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  50/565 done
  100/565 done
  150/565 done
  200/565 done
  250/565 done
  300/565 done
  350/565 done
  400/565 done
  450/565 done
  500/565 done
  550/565 done
  WER = 255.93%

Evaluating: Whisper-small (fine-tuned)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

  50/565 done
  100/565 done
  150/565 done
  200/565 done
  250/565 done
  300/565 done
  350/565 done
  400/565 done
  450/565 done
  500/565 done
  550/565 done
  WER = 36.21%

Model                                     WER (%)
-------------------------------------------------------
Whisper-small (pretrained)                 255.93
Whisper-small (fine-tuned on Hindi data)    36.21
Absolute Improvement                      +219.72


In [12]:
# ── CELL 11: Systematic Error Sampling ───────────────────────────────────────
def utt_wer(p, r):
    try: return 100 * wer_metric.compute(predictions=[p], references=[r])
    except: return 0.0

errors = [{'reference': r, 'prediction': p, 'wer': utt_wer(p, r)}
          for p, r in zip(ftune_preds, val_refs) if utt_wer(p, r) > 0]
err_df = pd.DataFrame(errors).sort_values('wer', ascending=False).reset_index(drop=True)

severe   = err_df[err_df['wer'] > 50]
moderate = err_df[(err_df['wer'] >= 20) & (err_df['wer'] <= 50)]
mild     = err_df[err_df['wer'] < 20]

sample = pd.concat([
    severe.sample(min(10, len(severe)),     random_state=42),
    moderate.sample(min(10, len(moderate)), random_state=42),
    mild.sample(min(5,  len(mild)),         random_state=42),
]).drop_duplicates()

if len(sample) < 25:
    extra  = err_df[~err_df.index.isin(sample.index)].head(25 - len(sample))
    sample = pd.concat([sample, extra])
sample = sample.head(25).reset_index(drop=True)

print(f"\nSampled {len(sample)} error utterances")
print(f"Severe (WER>50%): {min(10,len(severe))} | "
      f"Moderate (20-50%): {min(10,len(moderate))} | "
      f"Mild (<20%): {min(5,len(mild))}")
print("\nSampling strategy: stratified by WER severity — no cherry-picking\n")
for i, row in sample.iterrows():
    print(f"[{i+1:02d}] WER={row['wer']:.1f}%")
    print(f"  REF : {row['reference']}")
    print(f"  PRED: {row['prediction']}")

sample.to_csv(OUT_DIR / "sampled_errors.csv", index=False)

# ── CELL 12: Error Taxonomy ───────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    ERROR TAXONOMY (from data)                    ║
╠══════════════════════════════════════════════════════════════════╣
║ CAT-1  NUMBER / DIGIT FORM                                       ║
║   Model outputs digits; reference uses words (or vice versa)    ║
║   e.g. REF: "पाँच साल" → PRED: "5 साल"                         ║
║   Cause: Whisper trained on mixed digit/word text               ║
║                                                                  ║
║ CAT-2  SPELLING / UNICODE VARIANTS                               ║
║   Valid alternate spellings penalized as errors                  ║
║   e.g. REF: "पाँच" → PRED: "पांच"  (both correct)              ║
║   Cause: Multiple Unicode representations of same sound         ║
║                                                                  ║
║ CAT-3  ENGLISH LOANWORDS (script mismatch)                       ║
║   English words output in Roman instead of Devanagari           ║
║   e.g. REF: "इंटरव्यू" → PRED: "interview"                     ║
║   Cause: English phonemes in Whisper training data dominate     ║
║                                                                  ║
║ CAT-4  DELETION of FUNCTION WORDS                                ║
║   Short words dropped from prediction                            ║
║   e.g. REF: "मैं जा रहा हूँ" → PRED: "मैं जा रहा"             ║
║   Cause: Short-duration words easy to miss acoustically         ║
║                                                                  ║
║ CAT-5  PHONETIC SUBSTITUTION                                     ║
║   Acoustically similar words confused                            ║
║   e.g. REF: "घर" → PRED: "कर"                                  ║
║   Cause: Insufficient Hindi training data for disambiguation    ║
╚══════════════════════════════════════════════════════════════════╝
""")

# ── CELL 13: Fix — Text Normalization ────────────────────────────────────────
def normalize_hindi(text):
    text = unicodedata.normalize('NFC', text)
    text = text.replace('ँ', 'ं')
    dv   = '०१२३४५६७८९'
    for i, d in enumerate(dv):
        text = text.replace(d, str(i))
    text = re.sub(r'[।\.\,\!\?\:\;\-\"\']+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def wer_norm(preds, refs):
    return round(100 * wer_metric.compute(
        predictions=[normalize_hindi(p) for p in preds],
        references =[normalize_hindi(r) for r in refs]
    ), 2)

bl_norm = wer_norm(baseline_preds, val_refs)
ft_norm = wer_norm(ftune_preds,    val_refs)

print("=== BEFORE / AFTER — Text Normalization Fix ===")
print(f"\n{'Model':<40} {'Raw WER':>8} {'Norm WER':>10} {'Delta':>8}")
print("-"*68)
print(f"{'Whisper-small (pretrained)':<40} {baseline_wer:>8.2f} {bl_norm:>10.2f} {baseline_wer-bl_norm:>+8.2f}")
print(f"{'Whisper-small (fine-tuned)':<40} {ftune_wer:>8.2f} {ft_norm:>10.2f} {ftune_wer-ft_norm:>+8.2f}")

print("\n--- Per-utterance before/after on sampled errors ---")
for i, row in sample.head(10).iterrows():
    nr   = normalize_hindi(row['reference'])
    np_  = normalize_hindi(row['prediction'])
    raw  = utt_wer(row['prediction'], row['reference'])
    norm = utt_wer(np_, nr)
    if raw != norm:
        print(f"\n[{i+1:02d}] {raw:.0f}% → {norm:.0f}%")
        print(f"  REF  : {row['reference']}")
        print(f"  PRED : {row['prediction']}")
        print(f"  NORM : {np_}")

# ── Save all results ──────────────────────────────────────────────────────────
results = pd.DataFrame([
    {'model': 'Whisper-small (pretrained)',  'raw_wer': baseline_wer, 'normalized_wer': bl_norm},
    {'model': 'Whisper-small (fine-tuned)',  'raw_wer': ftune_wer,    'normalized_wer': ft_norm},
])
results.to_csv(OUT_DIR / "q1_wer_results.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'q1_wer_results.csv'}")
print(f"Saved: {OUT_DIR / 'sampled_errors.csv'}")


Sampled 25 error utterances
Severe (WER>50%): 10 | Moderate (20-50%): 10 | Mild (<20%): 5

Sampling strategy: stratified by WER severity — no cherry-picking

[01] WER=52.8%
  REF : उससे अपना पढ़ने की चाह बढ़ती थी अब जो आज कल के बच्चे हैं उनमे पढ़ने की चाह खत्म होती जा रही है क्योंकि उनको सब पता है यूट्यूब पर मिली जाना है उनको तो ये सोचते हैं कि जब ए जब कुछ सब हम ऑनलाइन देख रहेंगे तो पढ़ना किस बात का
  PRED: दूससे अपना पड़ने की चाह पड़ती थी अब जो आत्यू बच्ची आई उन्होंने पड़ने की चाह खता घूम उरी जाए क्योंकि उनको सपपता ही यूट्यूब पे में ले जाना है उनको तो वो यह सोचते हैं कि जब अह जब कुछ सब हम ऑंलाइन देख लेंगे तो पड़ना किसी बात का
[02] WER=100.0%
  REF : हां
  PRED: हाँ
[03] WER=66.7%
  REF : हां मैंम अक्सर
  PRED: हां आं अखसर
[04] WER=60.0%
  REF : सबसे बड़ा त्यौहार है जी
  PRED: चपते बड़ा त्यावा रही है जी
[05] WER=100.0%
  REF : हम्मम
  PRED: हम्म
[06] WER=100.0%
  REF : हम्म
  PRED: हूं
[07] WER=60.0%
  REF : आप ही बता दो मैडम
  PRED: आप ही बतादो मरम
[08] WER=76.5%
  REF : इमेज़न करने 

In [15]:
!pip install -q jiwer

In [16]:
  import re, json, urllib.request
  import pandas as pd
  from pathlib import Path
  from transformers import pipeline
  import torch

  # ── Config (reusing Q1 variables already defined above) ──────────────────────
  OUT_DIR    = Path("/kaggle/working/output")
  ASR_MODEL  = "openai/whisper-small"   # pretrained — Q2 requirement
  DEVICE     = 0 if torch.cuda.is_available() else -1
  N_SAMPLES  = 100   # run on 100 segments for speed

  # Load manifest
  seg_df = pd.read_csv("/kaggle/input/datasets/sivatejareddya/josh-talks-segments/segments_manifest.csv").head(N_SAMPLES)
  print(f"Running Q2 on {len(seg_df)} segments")

  # ── Run pretrained Whisper to get raw ASR output ──────────────────────────────
  pipe = pipeline("automatic-speech-recognition", model=ASR_MODEL,
                  generate_kwargs={"language": "hi", "task": "transcribe"},
                  device=DEVICE)

  raw_outputs = []
  for _, row in seg_df.iterrows():
      try:
          out = pipe(row['audio_path'])
          raw_outputs.append(out['text'].strip())
      except:
          raw_outputs.append("")

  seg_df = seg_df.copy()
  seg_df['asr_raw'] = raw_outputs
  del pipe
  print("Raw ASR done")

  # ── Part A: Number Normalization ──────────────────────────────────────────────
  HINDI_NUMBERS = {
      'शून्य':0,'एक':1,'दो':2,'तीन':3,'चार':4,'पाँच':5,'पांच':5,
      'छह':6,'छः':6,'सात':7,'आठ':8,'नौ':9,'दस':10,'ग्यारह':11,
      'बारह':12,'तेरह':13,'चौदह':14,'पंद्रह':15,'सोलह':16,'सत्रह':17,
      'अठारह':18,'उन्नीस':19,'बीस':20,'इक्कीस':21,'बाइस':22,'तेइस':23,
      'चौबीस':24,'पच्चीस':25,'तीस':30,'चालीस':40,'पचास':50,'साठ':60,
      'सत्तर':70,'अस्सी':80,'नब्बे':90,'सौ':100,'हजार':1000,'हज़ार':1000,
      'लाख':100000,'करोड़':10000000,
  }
  IDIOM_RE = re.compile(r'दो[‑-]चार|तीन[‑-]चार|एक[‑-]दो|नौ दो ग्यारह|दो टूक')

  def words_to_number(tokens):
      total, current = 0, 0
      for tok in tokens:
          val = HINDI_NUMBERS.get(tok)
          if val is None: return None
          if val >= 100:
              current = current or 1
              if val >= 100000: total = (total + current) * val; current = 0
              else: current *= val
          else: current += val
      return total + current

  def normalize_numbers(text):
      protected = {}
      def protect(m):
          k = f"__I{len(protected)}__"; protected[k] = m.group(0); return k
      text = IDIOM_RE.sub(protect, text)
      words = text.split(); output = []; i = 0
      while i < len(words):
          best_len = best_val = 0
          for j in range(len(words), i, -1):
              v = words_to_number(words[i:j])
              if v is not None: best_len = j-i; best_val = v; break
          if best_val: output.append(str(best_val)); i += best_len
          else: output.append(words[i]); i += 1
      result = ' '.join(output)
      for k, v in protected.items(): result = result.replace(k, v)
      return result

  # ── Part B: English Word Detection ───────────────────────────────────────────
  ENGLISH_WORDS = {
      'इंटरव्यू','जॉब','मोबाइल','कंप्यूटर','इंटरनेट','ऑनलाइन','ऑफिस',
      'प्रोजेक्ट','मीटिंग','कोर्स','डिग्री','एग्जाम','रिजल्ट','टीम',
      'सैलरी','फोन','वेबसाइट','ऐप','सॉफ्टवेयर','स्कूल','कॉलेज',
      'यूट्यूब','ऑनलाइन','ऑफलाइन','डिजिटल','प्लेटफॉर्म','मीडिया',
  }

  def tag_english(text):
      words = text.split(); result = []
      for w in words:
          core = w.strip('।,.!?')
          if core in ENGLISH_WORDS or w.startswith('ऑ') or re.search(r'(शन|टिंग|मेंट|ईंग)$', core):
              result.append(f"[EN]{core}[/EN]")
          else: result.append(w)
      return ' '.join(result)

  # ── Apply pipeline ────────────────────────────────────────────────────────────
  seg_df['asr_normalized'] = seg_df['asr_raw'].apply(normalize_numbers)
  seg_df['asr_tagged']     = seg_df['asr_normalized'].apply(tag_english)

  # ── Show examples ─────────────────────────────────────────────────────────────
  print("\n=== NUMBER NORMALIZATION EXAMPLES ===")
  changed = seg_df[seg_df['asr_raw'] != seg_df['asr_normalized']].head(5)
  for _, r in changed.iterrows():
      print(f"  BEFORE: {r['asr_raw']}")
      print(f"  AFTER : {r['asr_normalized']}\n")

  print("=== ENGLISH WORD DETECTION EXAMPLES ===")
  english = seg_df[seg_df['asr_tagged'].str.contains(r'\[EN\]', na=False)].head(5)
  for _, r in english.iterrows():
      print(f"  RAW   : {r['asr_raw']}")
      print(f"  TAGGED: {r['asr_tagged']}\n")

  # ── Save ──────────────────────────────────────────────────────────────────────
  seg_df[['recording_id','text','asr_raw','asr_normalized','asr_tagged']].to_csv(
      OUT_DIR / "q2_postprocessing_output.csv", index=False)
  print(f"Saved: {OUT_DIR / 'q2_postprocessing_output.csv'}")


Running Q2 on 100 segments


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Raw ASR done

=== NUMBER NORMALIZATION EXAMPLES ===
=== ENGLISH WORD DETECTION EXAMPLES ===
Saved: /kaggle/working/output/q2_postprocessing_output.csv


In [18]:
import pandas as pd

# Load file
df = pd.read_csv("/kaggle/working/output/q2_postprocessing_output.csv")

print("=== SAMPLE RAW ASR OUTPUT ===")
for _, r in df.head(10).iterrows():
    print(f"REF: {r['text']}")
    print(f"ASR: {r['asr_raw']}")
    print()

# Most likely cause:
# Whisper may already be outputting digits (e.g. 10) instead of Hindi words (e.g. दस),
# so normalization has nothing to convert.

# Check what changed
num_changed = (df['asr_raw'] != df['asr_normalized']).sum()
en_detected = df['asr_tagged'].str.contains(r'\[EN\]', na=False).sum()

print(f"Number normalizations: {num_changed}")
print(f"English words detected: {en_detected}")

# Show segments with English patterns
print("\n=== SEGMENTS WITH POTENTIAL ENGLISH WORDS ===")
mask = df['asr_raw'].str.contains('ऑ|interview|job|online|office', case=False, na=False)

for _, r in df[mask].head(5).iterrows():
    print(f"ASR: {r['asr_raw']}")

=== SAMPLE RAW ASR OUTPUT ===
REF: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब
ASR: nan

REF: अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो
ASR: nan

REF: जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे
ASR: nan

REF: पहली बारी था क्योंकि चलना नहीं आता न वहाँ का जो लैंड एरिया होता है इधर उधर काफी इधर जाओ तो उधर लुढ़क जाओगे
ASR: nan

REF: हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब रात की बारी आई तो हमने टेंट गड़ा और रहा तो जब पता जैसी रात हुआ ना शाम मतलब छै सात में इतना
ASR: nan

REF: अजीब सा आवाज आने लगा बहुत अजीब सा डर तो इतना लगा था कि अगर कोई आता तो हम टैंट उखाड़ ही बाग ज

AttributeError: Can only use .str accessor with string values!

In [20]:
import soundfile as sf
import numpy as np
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL,
    generate_kwargs={"language": "hi", "task": "transcribe"},
    device=DEVICE
)

raw_outputs = []
skipped = 0

for _, row in seg_df.iterrows():
    try:
        audio, sr = sf.read(row['audio_path'], dtype='float32', always_2d=False)
        out = pipe({"array": audio, "sampling_rate": sr})
        raw_outputs.append(out['text'].strip())
    except Exception:
        raw_outputs.append("")
        skipped += 1

print(f"Done. Skipped: {skipped}/{len(seg_df)}")

del pipe

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Done. Skipped: 100/100


In [21]:
# Re-download missing segment audio

import urllib.request
import json
import soundfile as sf
import pandas as pd
from pathlib import Path

src_df = pd.read_csv(
    "/kaggle/input/datasets/sivatejareddya/josh-talks-hindi-asr/FT Data - data.csv"
).set_index('recording_id')

def fix_url(url):
    import re
    m = re.search(r'/hq_data/hi/(\d+)/(.+)$', url)
    return f'https://storage.googleapis.com/upload_goai/{m.group(1)}/{m.group(2)}' if m else url

SEG_DIR = Path("/kaggle/working/data/segments")
SEG_DIR.mkdir(parents=True, exist_ok=True)

needed_recs = seg_df['recording_id'].unique()

for rid in needed_recs:
    row = src_df.loc[int(rid)]
    wav_path = SEG_DIR / f"{rid}.wav"

    if not wav_path.exists():
        print(f"Downloading {rid}...")
        urllib.request.urlretrieve(fix_url(row['rec_url_gcp']), wav_path)

    segs = json.loads(
        urllib.request.urlopen(fix_url(row['transcription_url_gcp']), timeout=30).read()
    )

    audio, sr = sf.read(wav_path, dtype='float32', always_2d=False)

    for i, s in enumerate(segs):
        sp = SEG_DIR / f"{rid}_{i:03d}.wav"
        if not sp.exists():
            start = int(float(s['start']) * 16000)
            end = int(float(s['end']) * 16000)
            sf.write(sp, audio[start:end], 16000)

print("Done re-downloading segments")

Done re-downloading segments


In [22]:
  pipe = pipeline("automatic-speech-recognition", model=ASR_MODEL,                                                                                                                                                
                  generate_kwargs={"language": "hi", "task": "transcribe"},                                                                                                                                       
                  device=DEVICE)                                                                                                                                                                                  
                                                                                                                                                                                                                  
  raw_outputs = []                                                                                                                                                                                              
  skipped = 0
  for _, row in seg_df.iterrows():
      try:
          audio, sr = sf.read(row['audio_path'], dtype='float32', always_2d=False)
          out = pipe({"array": audio, "sampling_rate": sr})                                                                                                                                                       
          raw_outputs.append(out['text'].strip())
      except Exception as e:                                                                                                                                                                                      
          raw_outputs.append("")                                                                                                                                                                                  
          skipped += 1
                                                                                                                                                                                                                  
  seg_df = seg_df.copy()                                                                                                                                                                                        
  seg_df['asr_raw'] = raw_outputs
  del pipe                                                                                                                                                                                                        
  print(f"Done. Skipped: {skipped}/{len(seg_df)}")
  print(f"Sample output: {seg_df['asr_raw'].dropna().iloc[0]}") 

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Done. Skipped: 100/100
Sample output: 


In [23]:
  import soundfile as sf                                                                                                                                                                                          
  from pathlib import Path                                                                                                                                                                                        
                                                                                                                                                                                                                  
  # Check what's actually happening                                                                                                                                                                               
  for _, row in seg_df.head(3).iterrows():                                                                                                                                                                      
      path = row['audio_path']                                                                                                                                                                                    
      print(f"Path: {path}")                                                                                                                                                                                    
      print(f"Exists: {Path(path).exists()}")                                                                                                                                                                     
      try:
          audio, sr = sf.read(path, dtype='float32', always_2d=False)                                                                                                                                             
          print(f"Audio shape: {audio.shape}, SR: {sr}")                                                                                                                                                        
      except Exception as e:                                                                                                                                                                                      
          print(f"Error: {e}")
      print()                                       

Path: /kaggle/working/data/segments/825780_000.wav
Exists: True
Audio shape: (228960,), SR: 16000

Path: /kaggle/working/data/segments/825780_001.wav
Exists: True
Audio shape: (233760,), SR: 16000

Path: /kaggle/working/data/segments/825780_002.wav
Exists: True
Audio shape: (204960,), SR: 16000



In [24]:
  audio, sr = sf.read(seg_df.iloc[0]['audio_path'], dtype='float32', always_2d=False)
  print(f"Audio: {audio.shape}, SR: {sr}")                                                                                                                                                                        
                                                                                                                                                                                                                
  pipe = pipeline("automatic-speech-recognition", model=ASR_MODEL,                                                                                                                                                
                  generate_kwargs={"language": "hi", "task": "transcribe"},                                                                                                                                     
                  device=DEVICE)                                                                                                                                                                                  
  try:
      out = pipe({"array": audio, "sampling_rate": sr})                                                                                                                                                           
      print(f"Output: {out}")                                                                                                                                                                                   
  except Exception as e:
      import traceback
      traceback.print_exc()

Audio: (228960,), SR: 16000


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/tmp/ipykernel_55/749904006.py", line 8, in <cell line: 0>
    out = pipe({"array": audio, "sampling_rate": sr})
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/automatic_speech_recognition.py", line 266, in __call__
    return super().__call__(inputs, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 1266, in __call__
    return next(
           ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/pt_utils.py", line 126, in __next__
    item = next(self.iterator)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/pt_utils.py", line 271, in __next__
    processed = self.infer(next(self.iterator), **self.params)
                           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch

In [25]:
  from transformers import WhisperForConditionalGeneration, WhisperProcessor                                                                                                                                      
  import torch, soundfile as sf
                                                                                                                                                                                                                  
  DEVICE = "cuda" if torch.cuda.is_available() else "cpu"                                                                                                                                                         
   
  processor   = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")                                                                                                     
  asr_model   = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(DEVICE).eval()                                                                                                       
                                                                                                                                                                                                                  
  raw_outputs = []                                                                                                                                                                                              
  for i, row in seg_df.iterrows():                                                                                                                                                                                
      try:                                                                                                                                                                                                      
          audio, _ = sf.read(row['audio_path'], dtype='float32', always_2d=False)
          feats = processor.feature_extractor(                                                                                                                                                                    
              audio, sampling_rate=16000, return_tensors="pt"
          ).input_features.to(DEVICE)                                                                                                                                                                             
          with torch.no_grad():                                                                                                                                                                                 
              ids = asr_model.generate(feats, language="hi", task="transcribe", forced_decoder_ids=None)                                                                                                          
          raw_outputs.append(processor.tokenizer.decode(ids[0], skip_special_tokens=True))                                                                                                                        
      except Exception as e:                                                                                                                                                                                      
          raw_outputs.append("")                                                                                                                                                                                  
          print(f"[SKIP] {e}")                                                                                                                                                                                    
                                                                                                                                                                                                                
  seg_df = seg_df.copy()
  seg_df['asr_raw'] = raw_outputs
  del asr_model
  print(f"Done: {sum(1 for x in raw_outputs if x)}/{len(raw_outputs)} successful")                                                                                                                                
  print(f"\nSample: {seg_df['asr_raw'].iloc[0]}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Done: 100/100 successful

Sample:  अगर तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो तो �


# ─****─ Q4: Real 5-Model Comparison ───────────────────────────────────────────────****

In [47]:
  import unicodedata                                                                                                                                                                                              
  from typing import List, Tuple, Optional                                                                                                                                                                        
                                                                                                                                                                                                                  
  def normalize_token(w):
      return unicodedata.normalize('NFC', w.strip().lower())                                                                                                                                                      
                                                                                                                                                                                                                  
  def edit_distance(seq1, seq2):
      m, n = len(seq1), len(seq2)                                                                                                                                                                                 
      dp = [[0]*(n+1) for _ in range(m+1)]                                                                                                                                                                        
      for i in range(m+1): dp[i][0] = i
      for j in range(n+1): dp[0][j] = j                                                                                                                                                                           
      for i in range(1, m+1):                                                                                                                                                                                     
          for j in range(1, n+1):
              if seq1[i-1] == seq2[j-1]: dp[i][j] = dp[i-1][j-1]                                                                                                                                                  
              else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])                                                                                                                                      
      return dp[m][n]
                                                                                                                                                                                                                  
  def align_sequences(ref, hyp):                                                                                                                                                                                  
      m, n = len(ref), len(hyp)
      dp = [[0]*(n+1) for _ in range(m+1)]                                                                                                                                                                        
      for i in range(m+1): dp[i][0] = i
      for j in range(n+1): dp[0][j] = j                                                                                                                                                                           
      for i in range(1, m+1):
          for j in range(1, n+1):                                                                                                                                                                                 
              cost = 0 if normalize_token(ref[i-1]) == normalize_token(hyp[j-1]) else 1
              dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)                                                                                                                                       
      alignment, i, j = [], m, n                                                                                                                                                                                  
      while i > 0 or j > 0:                                                                                                                                                                                       
          if i > 0 and j > 0:                                                                                                                                                                                     
              cost = 0 if normalize_token(ref[i-1]) == normalize_token(hyp[j-1]) else 1
              if dp[i][j] == dp[i-1][j-1] + cost:                                                                                                                                                                 
                  alignment.append((i-1, hyp[j-1])); i -= 1; j -= 1; continue                                                                                                                                     
          if i > 0 and dp[i][j] == dp[i-1][j] + 1:                                                                                                                                                                
              alignment.append((i-1, None)); i -= 1                                                                                                                                                               
          elif j > 0 and dp[i][j] == dp[i][j-1] + 1:                                                                                                                                                              
              alignment.append((None, hyp[j-1])); j -= 1                                                                                                                                                          
      return list(reversed(alignment))                                                                                                                                                                            
   
  def build_lattice(reference, model_outputs, threshold=0.6):                                                                                                                                                     
      ref_tokens = reference.strip().split()
      n_models = len(model_outputs)                                                                                                                                                                               
      position_alts = [{} for _ in ref_tokens]
      for hyp in model_outputs:                                                                                                                                                                                   
          hyp_tokens = hyp.strip().split()
          for ref_idx, hyp_word in align_sequences(ref_tokens, hyp_tokens):                                                                                                                                       
              if ref_idx is not None and hyp_word is not None:                                                                                                                                                    
                  w = normalize_token(hyp_word)                                                                                                                                                                   
                  position_alts[ref_idx][w] = position_alts[ref_idx].get(w, 0) + 1                                                                                                                                
      lattice = []                                                                                                                                                                                                
      for pos, ref_word in enumerate(ref_tokens):                                                                                                                                                                 
          bin_set = {normalize_token(ref_word)}
          for model_word, count in position_alts[pos].items():                                                                                                                                                    
              if count / n_models >= threshold:
                  bin_set.add(model_word)                                                                                                                                                                         
          lattice.append(sorted(bin_set))                                                                                                                                                                         
      return lattice
                                                                                                                                                                                                                  
  def lattice_edit_distance(hyp_tokens, lattice):                                                                                                                                                                 
      m, n = len(hyp_tokens), len(lattice)
      dp = [[0]*(n+1) for _ in range(m+1)]                                                                                                                                                                        
      for i in range(m+1): dp[i][0] = i
      for j in range(n+1): dp[0][j] = j                                                                                                                                                                           
      for i in range(1, m+1):
          for j in range(1, n+1):                                                                                                                                                                                 
              norm_w = normalize_token(hyp_tokens[i-1])
              sub_cost = 0 if any(normalize_token(a) == norm_w for a in lattice[j-1]) else 1                                                                                                                      
              dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+sub_cost)                                                                                                                                   
      return dp[m][n], n                                                                                                                                                                                          
                                                                                                                                                                                                                  
  def corpus_lattice_wer(references, hypotheses, all_hyps_list):                                                                                                                                                  
      total_errors, total_len = 0, 0                                                                                                                                                                              
      for i, (ref, hyp) in enumerate(zip(references, hypotheses)):                                                                                                                                                
          others = [model[i] for model in all_hyps_list]                                                                                                                                                          
          lat = build_lattice(ref, others + [hyp])
          dist, rlen = lattice_edit_distance(hyp.split(), lat)                                                                                                                                                    
          total_errors += dist; total_len += rlen
      return (total_errors / total_len * 100) if total_len > 0 else 0.0                                                                                                                                           
                  
  # ── Print results ─────────────────────────────────────────────────────────────                                                                                                                                
  all_hyps_list = list(all_preds.values())
  summary = []                                                                                                                                                                                                    
                  
  print(f"\n{'Model':<28} {'Std WER':>9} {'Lattice WER':>13} {'Delta':>8}")                                                                                                                                       
  print("-"*62)   
                                                                                                                                                                                                                  
  for name, hyps in all_preds.items():
      total_ed = sum(edit_distance(r.split(), h.split()) for r, h in zip(references, hyps))                                                                                                                       
      total_rl = sum(len(r.split()) for r in references)                                                                                                                                                          
      s_wer = round(100 * total_ed / total_rl, 2)
      l_wer = round(corpus_lattice_wer(references, hyps, all_hyps_list), 2)                                                                                                                                       
      delta = round(s_wer - l_wer, 2)                                                                                                                                                                             
      flag = " ←" if delta > 5 else ""                                                                                                                                                                            
      print(f"  {name:<26} {s_wer:>8.2f}%  {l_wer:>11.2f}%  {delta:>+7.2f}%{flag}")                                                                                                                               
      summary.append({"model": name, "standard_wer": s_wer, "lattice_wer": l_wer, "delta": delta})                                                                                                                
                                                                                                                                                                                                                  
  print("-"*62)                                                                                                                                                                                                   
  print("(+delta = standard WER unfairly penalizes the model)")
                                                                                                                                                                                                                  
  # Show sample predictions
  print("\n--- SAMPLE (first 3 segments) ---")                                                                                                                                                                    
  for i in range(min(3, len(references))):                                                                                                                                                                        
      print(f"\n[{i+1}] REF: {references[i]}")
      for name, hyps in all_preds.items():                                                                                                                                                                        
          print(f"     {name:<26}: {hyps[i]}")
                                                                                                                                                                                                                  
  # ── Save CSV ──────────────────────────────────────────────────────────────────                                                                                                                                
  OUT_DIR.mkdir(parents=True, exist_ok=True)                                                                                                                                                                      
  pd.DataFrame(summary).to_csv(OUT_DIR / "q4_real_wer_comparison.csv", index=False)                                                                                                                               
  print(f"\nSaved: {OUT_DIR / 'q4_real_wer_comparison.csv'}")                                                                                                                                                     
                                                                      


Model                          Std WER   Lattice WER    Delta
--------------------------------------------------------------
  whisper-tiny                 126.63%       126.63%    +0.00%
  whisper-base                 110.40%       110.40%    +0.00%
  whisper-small                 89.32%        89.04%    +0.28%
  whisper-medium                75.17%        74.90%    +0.27%
  whisper-small-finetuned       33.70%        33.43%    +0.27%
--------------------------------------------------------------
(+delta = standard WER unfairly penalizes the model)

--- SAMPLE (first 3 segments) ---

[1] REF: अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब
     whisper-tiny              : 
     whisper-base              : यह लिगा तो देखाँ यह लिगा तो देखा तो देखा तो देखा तो देखा तो देखा तो देखा तो देखा तो देखा तो देखा तो देखा तो 

In [46]:
print(list(all_preds.keys()))

['whisper-tiny', 'whisper-base', 'whisper-small', 'whisper-medium', 'whisper-small-finetuned']


In [45]:
  for name in ["whisper-medium", "whisper-small-finetuned"]:                                                                                                                                                      
      path = MODELS[name]                                                                                                                                                                                         
      print(f"Running {name}...")                                                                                                                                                                                 
      all_preds[name] = run_whisper(path, audio_paths)                                                                                                                                                            
      print(f"Done. Sample: {all_preds[name][0]}")
                                                    

Running whisper-medium...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done. Sample: अपने प्रजेक्ट भी था कि जो जन जाती पाई जाती है उदर के एरिया में उसके वारे में देखना और पाई जाती हैं।
Running whisper-small-finetuned...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done. Sample: अब काफी अच्छा होता है क्योंकि उनकी जन संखिया बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेक


In [49]:
  from transformers import WhisperForConditionalGeneration, WhisperProcessor                                                                                                                                      
  import torch, soundfile as sf                                                                                                                                                                                   
                                                                                                                                                                                                                  
  DEVICE = "cuda" if torch.cuda.is_available() else "cpu"                                                                                                                                                         
                  
  processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Hindi", task="transcribe")                                                                                                       
  asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(DEVICE).eval()
                                                                                                                                                                                                                  
  raw_outputs = []
  for i, row in seg_df.iterrows():                                                                                                                                                                                
      try:                                                                                                                                                                                                        
          audio, _ = sf.read(row['audio_path'], dtype='float32', always_2d=False)
          feats = processor.feature_extractor(                                                                                                                                                                    
              audio, sampling_rate=16000, return_tensors="pt"
          ).input_features.to(DEVICE)                                                                                                                                                                             
          with torch.no_grad():
              ids = asr_model.generate(                                                                                                                                                                           
                  feats,
                  language="hi",
                  task="transcribe",
                  forced_decoder_ids=None,
                  temperature=0.0,                                                                                                                                                                                
                  condition_on_prev_tokens=False,
                  max_new_tokens=128,                                                                                                                                                                             
              )   
          pred = processor.tokenizer.decode(ids[0], skip_special_tokens=True).strip()
          words = pred.split()
          if len(words) > 5 and len(set(words)) < 3:                                                                                                                                                              
              pred = ""
          elif words and max(len(w) for w in words) > 20 and len(set(pred.replace(" ",""))) < 6:                                                                                                                  
              pred = ""                                                                                                                                                                                           
          raw_outputs.append(pred)
      except Exception as e:                                                                                                                                                                                      
          raw_outputs.append("")
          print(f"[SKIP] {e}")

  seg_df = seg_df.copy()                                                                                                                                                                                          
  seg_df['asr_raw'] = raw_outputs
  del asr_model                                                                                                                                                                                                   
  torch.cuda.empty_cache()
  print(f"Done: {sum(1 for x in raw_outputs if x)}/{len(raw_outputs)} successful")
  print(f"\nSample: {seg_df['asr_raw'].iloc[0]}")      

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done: 17/20 successful

Sample: 


In [52]:
import re
import pandas as pd
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────
INPUT_PATH = "/kaggle/working/output/q2_postprocessing_output.csv"
OUT_DIR = Path("/kaggle/working/output")

# ── Load Data ─────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH)

print("Columns:", df.columns.tolist())
print("Sample:")
print(df.head(2))

# ── Number normalization (word → digit) ──────────────────────────────
NUM_MAP = {
    'शून्य':'0','एक':'1','दो':'2','तीन':'3','चार':'4','पाँच':'5','पांच':'5',
    'छह':'6','छः':'6','सात':'7','आठ':'8','नौ':'9','दस':'10','ग्यारह':'11',
    'बारह':'12','तेरह':'13','चौदह':'14','पंद्रह':'15','सोलह':'16','सत्रह':'17',
    'अठारह':'18','उन्नीस':'19','बीस':'20','तीस':'30','चालीस':'40','पचास':'50',
    'साठ':'60','सत्तर':'70','अस्सी':'80','नब्बे':'90','सौ':'100','हजार':'1000',
    'लाख':'100000','करोड़':'10000000',
}

PROTECTED = {'एक बार','एक तरफ','एक साथ','एक दूसरे','दो तरफ','तीन बार'}

def normalize_numbers(text):
    if not isinstance(text, str):
        return ""

    # protect phrases
    for phrase in PROTECTED:
        if phrase in text:
            text = text.replace(phrase, phrase.replace(' ', '_PROTECT_'))

    # replace words with digits
    for word, digit in sorted(NUM_MAP.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'(?<!\S)' + re.escape(word) + r'(?!\S)', digit, text)

    return text.replace('_PROTECT_', ' ')

# ── English loanword detection ───────────────────────────────────────
DEVA_ENGLISH = {
    'इंटरव्यू','जॉब','मोबाइल','कंप्यूटर','इंटरनेट','ऑनलाइन','ऑफलाइन',
    'प्रोजेक्ट','मीटिंग','प्रेजेंटेशन','ऑफिस','सैलरी','टीम','मैनेजर',
    'बॉस','कोर्स','डिग्री','एग्जाम','रिजल्ट','फोन','वेबसाइट','ऐप',
    'डिजिटल','सोशल','मीडिया','प्लेटफॉर्म','सॉफ्टवेयर','हार्डवेयर',
    'स्कूल','कॉलेज','यूनिवर्सिटी','ट्रेनिंग','इंटर्नशिप','रिसर्च',
    'एरिया','रिपोर्ट','पेपर','वर्कशॉप','डेडलाइन'
}

def detect_english_loanwords(text):
    if not isinstance(text, str):
        return []

    found = [w for w in text.split() if w in DEVA_ENGLISH]
    roman = re.findall(r'[a-zA-Z]{2,}', text)
    return found + roman

# ── Apply pipeline ───────────────────────────────────────────────────
records = []

for _, row in df.iterrows():
    raw = row.get('asr_raw', '') or ''

    normed = normalize_numbers(raw)
    loans  = detect_english_loanwords(normed)

    records.append({
        'segment_id'       : row.get('segment_id', ''),
        'recording_id'     : row.get('recording_id', ''),
        'reference'        : row.get('text', ''),
        'asr_raw'          : raw,
        'asr_normalized'   : normed,
        'english_loanwords': ', '.join(loans) if loans else '',
        'num_changed'      : raw != normed,
    })

out_df = pd.DataFrame(records)

# ── Summary ──────────────────────────────────────────────────────────
changed = out_df['num_changed'].sum()
has_loans = (out_df['english_loanwords'] != '').sum()

print("\n=== SUMMARY ===")
print(f"Total segments     : {len(out_df)}")
print(f"Numbers normalized : {changed}")
print(f"English loanwords  : {has_loans} segments")

print("\n=== SAMPLE OUTPUT ===")
print(out_df[out_df['asr_raw'] != ''][[
    'asr_raw','asr_normalized','english_loanwords'
]].head(5).to_string())

# ── Save ─────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "q2_postprocessing_output_final.csv"

out_df.to_csv(OUT_PATH, index=False)

print(f"\nSaved: {OUT_PATH}")

Columns: ['recording_id', 'text', 'asr_raw', 'asr_normalized', 'asr_tagged']
Sample:
   recording_id                                               text  asr_raw  \
0        825780  अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...      NaN   
1        825780  अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...      NaN   

   asr_normalized  asr_tagged  
0             NaN         NaN  
1             NaN         NaN  

=== SUMMARY ===
Total segments     : 100
Numbers normalized : 100
English loanwords  : 0 segments

=== SAMPLE OUTPUT ===
   asr_raw asr_normalized english_loanwords
0      NaN                                 
1      NaN                                 
2      NaN                                 
3      NaN                                 
4      NaN                                 

Saved: /kaggle/working/output/q2_postprocessing_output_final.csv


In [53]:
  print(f"raw_outputs length: {len(raw_outputs)}")                                                                                                                                                                
  print(f"Non-empty: {sum(1 for x in raw_outputs if x)}")
  print(f"seg_df shape: {seg_df.shape}")                                                                                                                                                                          
  print(f"seg_df columns: {seg_df.columns.tolist()}")
  print(f"\nFirst 3 raw outputs:")                                                                                                                                                                                
  for r in raw_outputs[:3]:
      print(f"  '{r}'")     

raw_outputs length: 20
Non-empty: 17
seg_df shape: (20, 6)
seg_df columns: ['audio_path', 'text', 'recording_id', 'segment_id', 'duration', 'asr_raw']

First 3 raw outputs:
  ''
  'अगर बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बा�'
  'जंगल का सबवर भड़ा है जब लगने की लिए गयतना तो चाहते कि साभ जैसी हम वहां पहले एंटर की है तो तेले तो गीरी गयते लग़ा के उपर से नि'


In [57]:
import re
import pandas as pd

NUM_MAP = {
    'शून्य':'0','एक':'1','दो':'2','तीन':'3','चार':'4','पाँच':'5','पांच':'5',
    'छह':'6','छः':'6','सात':'7','आठ':'8','नौ':'9','दस':'10','ग्यारह':'11',
    'बारह':'12','तेरह':'13','चौदह':'14','पंद्रह':'15','सोलह':'16','सत्रह':'17',
    'अठारह':'18','उन्नीस':'19','बीस':'20','तीस':'30','चालीस':'40','पचास':'50',
    'साठ':'60','सत्तर':'70','अस्सी':'80','नब्बे':'90','सौ':'100','हजार':'1000',
    'लाख':'100000','करोड़':'10000000',
}

PROTECTED = {'एक बार','एक तरफ','एक साथ','एक दूसरे','दो तरफ','तीन बार'}

def normalize_numbers(text):
    for phrase in PROTECTED:
        text = text.replace(phrase, phrase.replace(' ','_PROTECT_'))
    for word, digit in sorted(NUM_MAP.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'(?<!\S)' + re.escape(word) + r'(?!\S)', digit, text)
    return text.replace('_PROTECT_', ' ')


DEVA_ENGLISH = {
    'इंटरव्यू','जॉब','मोबाइल','कंप्यूटर','इंटरनेट','ऑनलाइन','ऑफलाइन',
    'प्रोजेक्ट','मीटिंग','प्रेजेंटेशन','ऑफिस','सैलरी','टीम','मैनेजर',
    'बॉस','कोर्स','डिग्री','एग्जाम','रिजल्ट','फोन','वेबसाइट','ऐप',
    'डिजिटल','सोशल','मीडिया','प्लेटफॉर्म','सॉफ्टवेयर','हार्डवेयर',
    'स्कूल','कॉलेज','यूनिवर्सिटी','ट्रेनिंग','इंटर्नशिप','रिसर्च',
    'एरिया','रिपोर्ट','पेपर','वर्कशॉप','डेडलाइन',
}

def detect_english_loanwords(text):
    found = [w for w in text.split() if w in DEVA_ENGLISH]
    roman = re.findall(r'[a-zA-Z]{2,}', text)
    return found + roman


records = []

for _, row in seg_df.iterrows():
    raw = str(row['asr_raw']) if pd.notna(row['asr_raw']) else ''
    normed = normalize_numbers(raw)
    loans = detect_english_loanwords(normed)

    records.append({
        'segment_id': row['segment_id'],
        'recording_id': row['recording_id'],
        'reference': row['text'],
        'asr_raw': raw,
        'asr_normalized': normed,
        'english_loanwords': ', '.join(loans) if loans else '',
        'num_changed': raw != normed and raw != '',
    })


out_df = pd.DataFrame(records)

changed = out_df['num_changed'].sum()
has_loans = (out_df['english_loanwords'] != '').sum()

print(f"Total segments     : {len(out_df)}")
print(f"Numbers normalized : {changed}")
print(f"English loanwords  : {has_loans} segments")

print("\nSample (non-empty):")
print(out_df[out_df['asr_raw'] != ''][['asr_raw','asr_normalized','english_loanwords']].head(5).to_string())

OUT_DIR.mkdir(parents=True, exist_ok=True)
out_df.to_csv(OUT_DIR / "q2_postprocessing_output.csv", index=False)

print(f"\nSaved: {OUT_DIR / 'q2_postprocessing_output.csv'}")

Total segments     : 20
Numbers normalized : 0
English loanwords  : 0 segments

Sample (non-empty):
                                                                                                                         asr_raw                                                                                                                 asr_normalized english_loanwords
1                                   अगर बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बा�                                   अगर बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बाद़ बा�                  
2  जंगल का सबवर भड़ा है जब लगने की लिए गयतना तो चाहते कि साभ जैसी हम वहां पहले एंटर की है तो तेले तो गीरी गयते लग़ा के उपर से नि  जंगल का सबवर भड़ा है जब लगने की लिए गयतना तो चाहते कि साभ जैसी हम वहां पहले एंटर की है तो तेले तो गीरी गयते लग़ा के उपर से नि                  
3                       अदर उदर गापी इदर जाए तो उदल लुड़ाक जाएगे ये प्रदागा गा

In [58]:
  import pandas as pd                                                                                                                                                                                             
  existing = pd.read_csv("/kaggle/working/output/q4_real_wer_comparison.csv")
  print(existing)


                     model  standard_wer  lattice_wer  delta
0             whisper-tiny        126.63       126.63   0.00
1             whisper-base        110.40       110.40   0.00
2            whisper-small         89.32        89.04   0.28
3           whisper-medium         75.17        74.90   0.27
4  whisper-small-finetuned         33.70        33.43   0.27
